Contending double mutations (RT / NRTI)

For a double mutation `p1: wt1->mt1, p2: wt2->mt2`, its **contenders** are the double
mutations it is in conflict with: those that use one of its two positions but put a
*different* residue there -- `(p1: wt1->other, q)` or `(p2: wt2->other, q)` for any partner
position `q` in the protein. Only those are mutually exclusive with it, so a sequence can be
the gain-of-fitness sequence of only one of them. A double mutation that shares a position
with the *same* residue is not a contender: M41L-T215F, M41L-D67N and D67N-T215F all agree
with each other and can hold in the same sequence; M41L-T215F and M41L-T215Y cannot.

This notebook takes three pairs and, for each of them, measures against those contenders:

* the observed **bivariate marginal** (how often the two mutations are actually seen together),
* the number of **gain-of-fitness sequences** of the pair and of each contender, using the
  position-local rule only (`dE12` beats the wild type, both single mutants and every other
  residue combination at the same two positions) -- i.e. *without* requiring `dE12` to beat
  the contenders,
* how much those gain-of-fitness sets **overlap**, which is exactly the double counting that
  the contender constraint removes,
* and the share of the sequences that actually **carry** the double mutation that come out as
  gain of fitness.

Labels are written in the **unreduced (amino-acid) alphabet**: each reduced letter stands for
a group of amino acids, and the label names the most frequent one in that group (the rule
`functions.reduced_to_unreduced` uses); the reduced label and the full group are kept in the
tables next to it.

`V75M-F77L`, `M41L-T215F` and `D67G-K219G` are reverse transcriptase (NRTI) mutations, so this
runs on RT, positions 39-226. In integrase these labels are not mutations at all in the reduced
alphabet (F77L -> A77A, M41L -> A41A, D67G -> A67A: wild type and mutant land in the same group).
In the reduced alphabet K219G and K219E are the same mutation (C219A, group A at 219 = LPVYWAMTGHE),
so `D67G-K219G` is the `D67G-K219E` of the NRTI list.

In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "utilities" / "functions.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")

sys.path.append(str(repo_root))

import utilities.functions as functions

importlib.reload(functions)

RT_start_index = 39
RT_end_index = 226

data_root = repo_root / "ms0_5"

RT_all_seq = functions.read_seq(str(data_root / "RT" / "data" / "rt.reduce4.seq"))
with open(str(data_root / "RT" / "data" / "rt.consensus.reduce4.seq"), "r") as f:
    RT_consensus_seq = f.read().strip()
RT_redux = functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)
with open(str(data_root / "RT" / "data" / "rt.weights.txt"), "r") as f:
    RT_weights = np.array([float(line.strip()) for line in f])
assert len(RT_weights) == len(RT_all_seq), "Weights and sequences must have the same length."

alphabet = ["A", "B", "C", "D"]


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2].

    Same content as functions.load_J_dict but as an array instead of a ~1M-entry dict.
    The 5th slot on the last axis is an all-zero column standing in for out-of-alphabet
    characters (mirrors the J_dict.get(..., 0) default).
    """
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


RT_J = build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), RT_start_index, RT_end_index)

# the unreduced alignment, for amino-acid labels
RT_all_seq_unreduced = functions.read_seq(str(data_root / "RT" / "data" / "rt.fullseq"))
assert len(RT_all_seq_unreduced) == len(RT_all_seq)
print(f"RT: {len(RT_all_seq)} sequences, positions {RT_start_index}-{RT_end_index}")

In [ ]:
# ---------------------------------------------------------------------------
# Energies. Same machinery as DMC_sebset_freq2.6_c.ipynb:
#
#   S(p, x) = sum_{o not in {p1, p2}} J[p, o, x, seq[o]]      (background, pair excluded)
#   T(p, x) = sum_{o != p}            J[p, o, x, seq[o]]      (background, full)
#   M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
#   dE12 = M[wt1, wt2] - M[mt1, mt2]
#
# S gives the numbers for one fixed position pair; T is what lets a pair be compared against
# double mutations at *other* position pairs, since the partner position can be subtracted back
# out of the full background.
# ---------------------------------------------------------------------------

_AA_CODE = np.full(256, 4, dtype=np.uint8)  # anything outside ABCD -> the zero-coupling slot
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    """One-hot encode an alignment as (N, L*5) float32, column order (position, aa)."""
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    """S(p, x) for x in ABCD, for every sequence -> (N, 4)."""
    A = Jm[p].copy()
    A[list(excluded)] = 0.0
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def pair_energies(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    """dE1, dE2, dE12, p(DMC) and the plain gain-of-fitness flag, per sequence.

    gof_local[n] is the gain-of-fitness test on its own -- the double mutant beats the wild type
    (dE12 > 0) and both single mutants (dE12 > dE1, dE12 > dE2) -- with no comparison against any
    competing double mutation. That comparison is the contender constraint, and it is what this
    notebook quantifies; contenders at these same two positions are covered by contender_scan
    with q = the other position, exactly like contenders that use a third one.
    """
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)

    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    base = M[:, wt1, wt2]
    de1 = base - M[:, mt1, wt2]
    de2 = base - M[:, wt1, mt2]
    de12 = base - M[:, mt1, mt2]
    gof_local = (de1 < de12) & (de2 < de12) & (de12 > 0)

    Mf = M.reshape(S1.shape[0], 16)
    Z = -Mf
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    p_dmc = E[:, mt1 * 4 + mt2] / E.sum(axis=1)
    return de1, de2, de12, p_dmc, gof_local


def background_energies(onehot, Jm):
    """T[n, p, x] = sum_{o != p} J[p, o, x, seq[o]], shape (N, L, 4). One matmul, float64."""
    L = Jm.shape[0]
    W = np.asarray(Jm, dtype=np.float64).transpose(0, 2, 1, 3).reshape(L * 4, L * 5)
    return (np.asarray(onehot, dtype=np.float64) @ W.T).reshape(-1, L, 4)


def de12_from_T(T, Jm, codes, p, q, a, b, u_p, u_q):
    """dE12 of one double mutation (p: u_p -> a, q: u_q -> b), per sequence."""
    sp, sq = codes[:, p], codes[:, q]
    Jpq = np.asarray(Jm[p, q], dtype=np.float64)
    Jqp = np.asarray(Jm[q, p], dtype=np.float64)
    return ((T[:, p, u_p] - T[:, p, a]) + (T[:, q, u_q] - T[:, q, b])
            - (Jpq[u_p, sq] - Jpq[a, sq]) - (Jqp[u_q, sp] - Jqp[b, sp])
            + (Jpq[u_p, u_q] - Jpq[a, b]))


def contender_scan(T, Jm, codes, p, u_p, m_p, u, subset=None, chunk=1024):
    """One sweep over every double mutation that uses position p.

    Walks all (p: u_p -> a, q: u_q -> b) with q any other position and a, b any residues, and
    for each one asks the plain gain-of-fitness question of pair_energies: does its dE12 beat the
    wild type and both single mutants at (p, q)? Wild type is u_p at p and the consensus residue
    u[q] at the partner. That test also enforces a != u_p and b != u_q on its own, since a single
    mutant cannot beat itself.

    A genuine double mutation is a *contender* of a pair that mutates p to m_p only when it
    puts a different residue there, a not in {u_p, m_p}. a == m_p agrees with the pair at p, so
    the two can hold in the same sequence -- and that is also what keeps the pair itself out of
    its own comparison.

    Returns a dict of (L, 4, 4) arrays indexed [q, a, b]
        n_gof         sequences where (p, q, a, b) is gain of fitness (local rule)
        n_gof_dmc     ... and the sequence actually carries a at p and b at q
        n_gof_subset  ... and the sequence is in `subset` (pass the target pair's gof mask to
                          get the overlap between the two gain-of-fitness sets)
        genuine, contender   which (q, a, b) are double mutations / are in conflict with the pair
    plus per-sequence arrays
        best, best_idx   the strongest contender, best_idx being q * 16 + a * 4 + b
        n_gof_here       how many contenders are gain of fitness in that sequence
    """
    N, L, _ = T.shape
    qs = np.arange(L)
    aa = np.arange(4)
    Jp = np.asarray(Jm[p], dtype=np.float64)     # (L, 4, 5), J[p, q, a, aa_at_q]
    Jq = np.asarray(Jm[:, p], dtype=np.float64)  # (L, 4, 5), J[q, p, b, aa_at_p]
    direct = Jp[qs, u_p, u][:, None, None] - Jp[:, :, :4]          # (L, 4, 4)

    genuine = np.ones((L, 4, 4), dtype=bool)
    genuine[p] = False             # q must be a different position
    genuine[:, u_p, :] = False     # a must be a mutation at p
    genuine[qs, :, u] = False      # b must be a mutation at q
    contender = genuine.copy()
    contender[:, m_p, :] = False   # a == m_p agrees with the pair at p -> compatible, not a rival
    contender_flat = contender.reshape(L * 16)

    n_gof = np.zeros(L * 16, dtype=np.int64)
    n_gof_dmc = np.zeros(L * 16, dtype=np.int64)
    n_gof_subset = np.zeros(L * 16, dtype=np.int64)
    best = np.empty(N)
    best_idx = np.empty(N, dtype=np.int64)
    n_gof_here = np.zeros(N, dtype=np.int64)

    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        cb, Ts = codes[s:e], T[s:e]
        sp = cb[:, p]
        dp = Ts[:, p, u_p][:, None] - Ts[:, p, :]                          # (B, 4)
        dq = np.take_along_axis(Ts, u[None, :, None], axis=2) - Ts         # (B, L, 4)
        cA = (Jp[qs[None, :], u_p, cb][:, :, None]
              - Jp[qs[None, :, None], aa[None, None, :], cb[:, :, None]])  # (B, L, 4)
        cB = (Jq[qs, u, sp[:, None]][:, :, None]
              - Jq[qs[None, :, None], aa[None, None, :], sp[:, None, None]])
        dE = (dp[:, None, :, None] + dq[:, :, None, :]
              - cA[:, :, :, None] - cB[:, :, None, :] + direct[None])      # (B, L, 4, 4)

        # --- plain gain of fitness: beat the wild type and both single mutants at (p, q) ---
        single_p = np.take_along_axis(dE, u[None, :, None, None].astype(np.int64), axis=3)
        single_q = dE[:, :, u_p, :][:, :, None, :]
        gwin = (dE > 0) & (dE > single_p) & (dE > single_q)     # (B, L, 4, 4)
        n_gof += gwin.sum(axis=0).reshape(L * 16)
        carry = ((sp[:, None, None, None] == aa[None, None, :, None])      # a at p
                 & (cb[:, :, None, None] == aa[None, None, None, :]))        # b at q

        n_gof_dmc += (gwin & carry).sum(axis=0).reshape(L * 16)
        if subset is not None:
            n_gof_subset += (gwin & subset[s:e, None, None, None]).sum(axis=0).reshape(L * 16)
        n_gof_here[s:e] = (gwin & contender[None]).sum(axis=(1, 2, 3))

        # --- strongest contender, for the "must beat every contender" rule ---
        flat = dE.reshape(e - s, L * 16)   # a view of dE; the -inf writes below destroy f16
        flat[:, ~contender_flat] = -np.inf
        rows = np.arange(e - s)
        i1 = flat.argmax(axis=1)
        best[s:e] = flat[rows, i1]
        best_idx[s:e] = i1

    shape = (L, 4, 4)
    return {'n_gof': n_gof.reshape(shape), 'n_gof_dmc': n_gof_dmc.reshape(shape),
            'n_gof_subset': n_gof_subset.reshape(shape), 'genuine': genuine,
            'contender': contender, 'best': best, 'best_idx': best_idx,
            'n_gof_here': n_gof_here}


def bivariate_marginals(codes, weights, p, L):
    """Observed counts and weight sums of (a at p, b at q) -> two (4, L, 4) arrays [a, q, b]."""
    cnt = np.zeros((4, L, 4))
    wcnt = np.zeros((4, L, 4))
    off = np.arange(L) * 5
    for a in range(4):
        m = codes[:, p] == a
        if not m.any():
            continue
        keys = (codes[m].astype(np.int64) + off).ravel()
        c = np.bincount(keys, minlength=L * 5).reshape(L, 5)
        w = np.bincount(keys, weights=np.repeat(weights[m], L), minlength=L * 5).reshape(L, 5)
        cnt[a], wcnt[a] = c[:, :4], w[:, :4]
    return cnt, wcnt

In [ ]:
# --- amino-acid labels -----------------------------------------------------------------
# Each reduced letter stands for a group of amino acids. A label names the most frequent
# amino acid of the group in the unreduced alignment -- the rule functions.reduced_to_unreduced
# uses, with the per-position counts taken once so it can be called for every table row.
_AA_ARR = np.frombuffer("".join(RT_all_seq_unreduced).encode(), dtype=np.uint8).reshape(
    len(RT_all_seq_unreduced), -1)
_AA_COUNT = np.zeros((_AA_ARR.shape[1], 128), dtype=np.int64)
for _c in range(ord('A'), ord('Z') + 1):
    _AA_COUNT[:, _c] = (_AA_ARR == _c).sum(axis=0)


def aa_group(redux, pos, letter):
    """The amino acids a reduced letter stands for at this position."""
    return "".join(redux.get((pos, letter), []))


def aa_of(redux, pos, letter):
    """The most frequent amino acid of that reduced group at that position.

    Ties go to the first one listed in the group, so a label is stable across runs;
    functions.reduced_to_unreduced resolves them through `set` order, which is not.
    """
    group = redux.get((pos, letter), [])
    return max(group, key=lambda x: _AA_COUNT[pos - RT_start_index, ord(x)], default='-')


def unreduced_pair(redux, reduced):
    """'D41C-D215B' -> 'M41L-T215F'."""
    out = []
    for part in reduced.split('-'):
        wt, pos, mt = part[0], int(part[1:-1]), part[-1]
        out.append(f"{aa_of(redux, pos, wt)}{pos}{aa_of(redux, pos, mt)}")
    return '-'.join(out)


def contender_report(pair, min_pos, all_seq, consensus_seq, redux, weights, J, top_n=12):
    """Everything this notebook asks about one pair, against every contender of that pair.

    Returns (summary, table). `table` has one row per double mutation: the pair itself first,
    then the contenders -- double mutations that want a different residue at one of the pair's
    two positions -- that either take gain of fitness away from it, or are frequently observed,
    or are frequent gain-of-fitness pairs themselves.
    """
    L = J.shape[0]
    N = len(all_seq)
    W = float(weights.sum())
    codes, onehot = encode_seqs(all_seq, min_pos, min_pos + L - 1)
    cons_codes, _ = encode_seqs([consensus_seq], min_pos, min_pos + L - 1)
    u = cons_codes[0]
    T = background_energies(onehot, J)

    p1_lab, p2_lab = functions.split_pairs(pair)
    wt1, pos1, mt1 = functions.split_pair(functions.unreduced_to_reduced(redux, p1_lab))
    wt2, pos2, mt2 = functions.split_pair(functions.unreduced_to_reduced(redux, p2_lab))
    p1, p2 = pos1 - min_pos, pos2 - min_pos
    iwt1, imt1 = "ABCD".index(wt1), "ABCD".index(mt1)
    iwt2, imt2 = "ABCD".index(wt2), "ABCD".index(mt2)
    assert imt1 != iwt1 and imt2 != iwt2, f"{pair} is not a double mutation in the reduced alphabet"
    for pp, ww in ((p1, iwt1), (p2, iwt2)):
        assert u[pp] == ww, f"{pair}: labelled wild type differs from the consensus at {pp + min_pos}"

    de1, de2, de12, p_SH, gof = pair_energies(onehot, J, p1, p2, iwt1, imt1, iwt2, imt2)
    assert np.array_equal(gof, (de1 < de12) & (de2 < de12) & (de12 > 0) & gof)  # local rule
    own = de12_from_T(T, J, codes, p1, p2, imt1, imt2, iwt1, iwt2)
    dmc = (codes[:, p1] == imt1) & (codes[:, p2] == imt2)

    scans = {p1: contender_scan(T, J, codes, p1, iwt1, imt1, u, subset=gof),
             p2: contender_scan(T, J, codes, p2, iwt2, imt2, u, subset=gof)}
    marg = {p1: bivariate_marginals(codes, weights, p1, L),
            p2: bivariate_marginals(codes, weights, p2, L)}

    # the pair has to beat the strongest contender at both of its positions
    gof_strict = gof & (own > scans[p1]['best']) & (own > scans[p2]['best'])

    # how many gain-of-fitness sequences of this pair are shared with a contender
    n_rival_gof = scans[p1]['n_gof_here'] + scans[p2]['n_gof_here']
    shared = gof & (n_rival_gof > 0)

    def row(p, q, a, b, beats=None):
        """One line of the table for the double mutation (p: u_p -> a, q: u_q -> b)."""
        sc, (cnt, wcnt) = scans[p], marg[p]
        n_obs = int(cnt[a, q, b])
        lo, hi = ((p, a), (q, b)) if p < q else ((q, b), (p, a))
        reduced = (f"{'ABCD'[u[lo[0]]]}{lo[0] + min_pos}{'ABCD'[lo[1]]}"
                   f"-{'ABCD'[u[hi[0]]]}{hi[0] + min_pos}{'ABCD'[hi[1]]}")
        return {
            'pair_aa': unreduced_pair(redux, reduced),
            'pair': reduced,
            'mutant_group': f"{lo[0] + min_pos}:{aa_group(redux, lo[0] + min_pos, 'ABCD'[lo[1]])}"
                            f" {hi[0] + min_pos}:{aa_group(redux, hi[0] + min_pos, 'ABCD'[hi[1]])}",
            'shares_position': p + min_pos,
            'n_obs': n_obs,
            'obs_freq': n_obs / N,
            'w_obs_freq': float(wcnt[a, q, b]) / W,
            'n_gof': int(sc['n_gof'][q, a, b]),
            'n_gof_dmc': int(sc['n_gof_dmc'][q, a, b]),
            'pct_dmc_gof': 100.0 * sc['n_gof_dmc'][q, a, b] / n_obs if n_obs else np.nan,
            'gof_overlap_with_target': int(sc['n_gof_subset'][q, a, b]),
            'n_beats_target': beats,
            '_key': (lo, hi),
        }

    # ---- pick the contenders worth listing -------------------------------------------
    keys = {}
    for p in (p1, p2):
        sc = scans[p]
        con = sc['contender']
        flat = con.reshape(-1)
        # contenders that actually take gain of fitness away from the pair
        beaten = gof & (own <= sc['best'])
        for idx, n in zip(*np.unique(sc['best_idx'][beaten], return_counts=True)):
            idx = int(idx)
            if flat[idx]:
                key = (p, idx // 16, (idx % 16) // 4, idx % 4)
                keys[key] = keys.get(key, 0) + int(n)
        # plus the most observed and the most gain-of-fitness contenders
        for arr in (marg[p][0].transpose(1, 0, 2), sc['n_gof']):   # both indexed [q, a, b]
            flat_arr = np.where(con, arr, -1).reshape(-1)
            for idx in np.argsort(-flat_arr)[:top_n]:
                if flat_arr[idx] > 0:
                    idx = int(idx)
                    keys.setdefault((p, idx // 16, (idx % 16) // 4, idx % 4), 0)

    rows = [row(p1, p2, imt1, imt2)]
    rows[0]['shares_position'] = 'target'
    rows[0]['n_beats_target'] = 0
    seen = {rows[0]['_key']}
    for (p, q, a, b), beats in sorted(keys.items(), key=lambda kv: -kv[1]):
        r = row(p, q, a, b, beats)
        if r['_key'] in seen:          # the same double mutation reached from the other position
            continue
        seen.add(r['_key'])
        rows.append(r)

    table = pd.DataFrame(rows).drop(columns='_key')
    table = pd.concat([table.iloc[:1],
                       table.iloc[1:].sort_values(['n_beats_target', 'n_obs'], ascending=False)])
    table = table.reset_index(drop=True)

    n_dmc = int(dmc.sum())
    summary = {
        'pair': pair,
        'reduced': f"{wt1}{pos1}{mt1}-{wt2}{pos2}{mt2}",
        'n_seqs': N,
        'n_obs': n_dmc,
        'obs_freq': n_dmc / N,
        'w_obs_freq': float(weights[dmc].sum()) / W,
        'n_gof': int(gof.sum()),
        'n_gof_dmc': int((gof & dmc).sum()),
        'pct_dmc_gof': 100.0 * (gof & dmc).sum() / n_dmc if n_dmc else np.nan,
        'pct_gof_dmc': 100.0 * (gof & dmc).sum() / gof.sum() if gof.sum() else np.nan,
        'n_gof_shared_with_contender': int(shared.sum()),
        'pct_gof_shared': 100.0 * shared.sum() / gof.sum() if gof.sum() else np.nan,
        'mean_contender_gof_per_seq': float(n_rival_gof[gof].mean()) if gof.any() else np.nan,
        'max_contender_gof_per_seq': int(n_rival_gof[gof].max()) if gof.any() else 0,
        'n_gof_strict': int(gof_strict.sum()),
        'n_gof_lost_to_contenders': int(gof.sum() - gof_strict.sum()),
    }
    return summary, table

In [ ]:
TARGET_PAIRS = ['V75M-F77L', 'M41L-T215F', 'D67G-K219G']

summaries = []
tables = {}
for pair in TARGET_PAIRS:
    s, t = contender_report(pair, RT_start_index, RT_all_seq, RT_consensus_seq,
                            RT_redux, RT_weights, RT_J)
    summaries.append(s)
    tables[pair] = t
    print(f"{pair} done")

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.float_format', '{:.4g}'.format)

for pair in TARGET_PAIRS:
    print('=' * 110)
    print(f'{pair}   (row 0 is the pair itself, the rest are its contenders)')
    print('=' * 110)
    display(tables[pair])

In [ ]:
out = pd.concat([t.assign(target=pair) for pair, t in tables.items()], ignore_index=True)
out = out[['target'] + [c for c in out.columns if c != 'target']]
out.to_csv('contending_dm_pairs_RT.csv', index=False)
summary_df.to_csv('contending_dm_pairs_RT_summary.csv', index=False)
print('wrote contending_dm_pairs_RT.csv and contending_dm_pairs_RT_summary.csv')

## Gain-of-fitness sets of a pair and its contenders

The circles are **gain-of-fitness sequence sets** (position-local rule, no contender
constraint); the number outside each circle is how many sequences **carry** that double
mutation. Circle areas are schematic -- the sets can differ by a factor of ten and no
three-circle drawing can be area-true -- so every region is labelled with its exact count.

`contender_view(target, contenders)` builds everything for any double mutation, so a pair is
swapped in by changing one string: the target is an amino-acid label (`'M41L-T215F'`), and the
two contenders drawn beside it may be given as amino acids (`'M41L-T215Y'`) or as reduced
letters (`'D41C-D215C'`), or left out, in which case the two contenders that share the most
gain-of-fitness sequences with the target are used. Contenders are the double mutations that
want a *different* residue at one of the target's positions -- the only ones it competes with.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# light-mode chart tokens; the three series hues validate as a set on this surface
SURFACE, INK, INK_2 = '#fcfcfb', '#0b0b0b', '#52514e'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']

_L = RT_J.shape[0]
_codes, _onehot = encode_seqs(RT_all_seq, RT_start_index, RT_start_index + _L - 1)
_cons_codes, _ = encode_seqs([RT_consensus_seq], RT_start_index, RT_start_index + _L - 1)
_u = _cons_codes[0]
_T = background_energies(_onehot, RT_J)
_N = len(RT_all_seq)


def _parse_reduced(label):
    """'A67D-D215B' -> [(position index, mutant residue index, wild type residue index), ...]"""
    out = []
    for part in label.split('-'):
        out.append((int(part[1:-1]) - RT_start_index, "ABCD".index(part[-1]), "ABCD".index(part[0])))
    return out


def gof_and_carriers(label):
    """Gain-of-fitness mask (local rule) and carrier mask of one reduced double mutation."""
    (p, a, up), (q, b, uq) = _parse_reduced(label)
    *_, gof = pair_energies(_onehot, RT_J, p, q, up, a, uq, b)
    return gof, (_codes[:, p] == a) & (_codes[:, q] == b)


def _group_text(reduced):
    (p, a, _), (q, b, _) = _parse_reduced(reduced)
    return (f"{p + RT_start_index}:{aa_group(RT_redux, p + RT_start_index, 'ABCD'[a])} "
            f"{q + RT_start_index}:{aa_group(RT_redux, q + RT_start_index, 'ABCD'[b])}")


def contender_view(target, contenders=None):
    """Everything the two figures need for one double mutation -- swap the pair, swap the figures.

    target      amino-acid label of the pair, e.g. 'M41L-T215F' or 'V75M-F77L'
    contenders  the two drawn beside it, as amino acids ('M41L-T215Y') or reduced letters
                ('D41C-D215C'); by default the two contenders that share the most
                gain-of-fitness sequences with the target

    Returns a dict with the three sets and their labels, the seven Venn region counts, the full
    contender table, and how many contenders are gain of fitness in each of the target's
    gain-of-fitness sequences.
    """
    a_lab, b_lab = functions.split_pairs(target)
    wt1, pos1, mt1 = functions.split_pair(functions.unreduced_to_reduced(RT_redux, a_lab))
    wt2, pos2, mt2 = functions.split_pair(functions.unreduced_to_reduced(RT_redux, b_lab))
    reduced = f'{wt1}{pos1}{mt1}-{wt2}{pos2}{mt2}'
    (p1, m1, w1), (p2, m2, w2) = _parse_reduced(reduced)
    assert m1 != w1 and m2 != w2, f'{target} is not a double mutation in the reduced alphabet'

    gof, _carry = gof_and_carriers(reduced)
    own = de12_from_T(_T, RT_J, _codes, p1, p2, m1, m2, w1, w2)

    frames, scans = [], {}
    for p, u_p, m_p in ((p1, w1, m1), (p2, w2, m2)):
        sc = contender_scan(_T, RT_J, _codes, p, u_p, m_p, _u, subset=gof)
        scans[p] = sc
        cnt, wcnt = bivariate_marginals(_codes, RT_weights, p, _L)
        beats = np.bincount(sc['best_idx'][gof & (own <= sc['best'])],
                            minlength=_L * 16).reshape(_L, 4, 4)
        idx = np.flatnonzero(sc['contender'].reshape(-1))
        q, a, b = idx // 16, (idx % 16) // 4, idx % 4
        lab = [f"{'ABCD'[u_p]}{p + RT_start_index}{'ABCD'[ai]}"
               f"-{'ABCD'[_u[qi]]}{qi + RT_start_index}{'ABCD'[bi]}" for qi, ai, bi in zip(q, a, b)]
        frames.append(pd.DataFrame({
            'pair': ['-'.join(sorted(l.split('-'), key=lambda x: int(x[1:-1]))) for l in lab],
            'shares_position': p + RT_start_index,
            'n_obs': cnt[a, q, b].astype(int),
            'w_obs_freq': wcnt[a, q, b] / float(RT_weights.sum()),
            'n_gof': sc['n_gof'][q, a, b],
            'gof_overlap_with_target': sc['n_gof_subset'][q, a, b],
            'n_beats_target': beats[q, a, b],
        }))

    # a contender using *both* of the target's positions turns up in both scans -- the counts are
    # the same from either side, so keep one row and add up where it beats the target
    frame = (pd.concat(frames, ignore_index=True)
             .groupby('pair', as_index=False)
             .agg(shares_position=('shares_position',
                                   lambda v: 'both' if len(v) > 1 else v.iloc[0]),
                  n_obs=('n_obs', 'max'), w_obs_freq=('w_obs_freq', 'max'),
                  n_gof=('n_gof', 'max'),
                  gof_overlap_with_target=('gof_overlap_with_target', 'max'),
                  n_beats_target=('n_beats_target', 'sum'))
             .sort_values(['gof_overlap_with_target', 'n_beats_target', 'n_obs'], ascending=False)
             .reset_index(drop=True))
    frame.insert(0, 'pair_aa', [unreduced_pair(RT_redux, p) for p in frame['pair']])
    n_gof_target = int(gof.sum())
    frame.insert(4, 'obs_freq', frame['n_obs'] / _N)
    # the overlap read both ways: as a share of the target's gof set, and of the contender's own
    frame['pct_of_target_gof'] = (100 * frame['gof_overlap_with_target'] / n_gof_target
                                  if n_gof_target else np.nan)
    frame['pct_of_own_gof'] = np.where(frame['n_gof'] > 0,
                                       100 * frame['gof_overlap_with_target']
                                       / frame['n_gof'].replace(0, np.nan), np.nan)

    if contenders is None:
        picked = list(frame['pair'].iloc[:2])
    else:
        picked = []
        for c in contenders:
            hit = frame[(frame['pair'] == c) | (frame['pair_aa'] == c)]
            if hit.empty:
                raise ValueError(f'{c} is not a contender of {target} '
                                 f'(it agrees with it at every shared position, or does not exist)')
            picked.append(hit['pair'].iloc[0])

    labels = [reduced] + picked
    masks = [gof_and_carriers(l) for l in labels]
    A, B, C = (m[0] for m in masks)
    regions = {'A': A & ~B & ~C, 'B': ~A & B & ~C, 'AB': A & B & ~C, 'C': ~A & ~B & C,
               'AC': A & ~B & C, 'BC': ~A & B & C, 'ABC': A & B & C}

    return {
        'target': target,
        'reduced': reduced,
        'labels': labels,
        # the target keeps the label it was asked for (K219G and K219E are the same reduced
        # mutation, so deriving it would rename the pair); contenders get the derived one
        'labels_aa': [target] + [unreduced_pair(RT_redux, l) for l in labels[1:]],
        'groups': [_group_text(l) for l in labels],
        'gof_masks': [m[0] for m in masks],
        'carrier_masks': [m[1] for m in masks],
        'region_n': {k: int(v.sum()) for k, v in regions.items()},
        'contenders': frame,
        'n_shared': (scans[p1]['n_gof_here'] + scans[p2]['n_gof_here'])[A],
    }


def export_contenders(view, path=None):
    """Write every contender of the target to CSV, the target itself on the first row.

    Columns: the pair as amino acids and as reduced letters, which of the target's positions it
    takes, its bivariate marginal (carriers over the whole alignment, as a count, a fraction and
    a weight fraction), its gain-of-fitness count under the plain rule (no contender constraint),
    how many of those sequences are also gain of fitness for the target, that overlap as a
    percentage of the target's gain-of-fitness set and of the contender's own, and in how many
    sequences it is the strongest rival that beats the target.
    """
    path = path or f"contenders_{view['target']}.csv"
    gof, carriers = view['gof_masks'][0], view['carrier_masks'][0]
    n_obs, n_gof = int(carriers.sum()), int(gof.sum())
    head = pd.DataFrame([{
        'pair_aa': view['target'], 'pair': view['reduced'], 'shares_position': 'target',
        'n_obs': n_obs, 'obs_freq': n_obs / _N,
        'w_obs_freq': float(RT_weights[carriers].sum()) / float(RT_weights.sum()),
        'n_gof': n_gof, 'gof_overlap_with_target': n_gof,
        'pct_of_target_gof': 100.0 if n_gof else np.nan,
        'pct_of_own_gof': 100.0 if n_gof else np.nan, 'n_beats_target': 0,
    }])
    out = pd.concat([head, view['contenders']], ignore_index=True)[head.columns]
    out.to_csv(path, index=False)
    print(f"wrote {path}: {len(out) - 1:,} contenders of {view['target']} "
          f"(+ the pair itself on row 1)")
    return out


def plot_venn(view, save=True):
    """Three gain-of-fitness sets, carriers labelled outside, exact counts in every region."""
    region_n, n_none = view['region_n'], _N - sum(view['region_n'].values())
    fig, ax = plt.subplots(figsize=(7.8, 7.4), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.84, bottom=0.02)

    fig.text(0.035, 0.975, f"Gain-of-fitness sequences of {view['target']} and two of its contenders",
             ha='left', va='top', color=INK, fontsize=13.5)
    fig.text(0.035, 0.935,
             'circles are gain-of-fitness sets (position-local rule, no contender constraint); the count\n'
             'outside each circle is how many sequences carry that double mutation; a label names the\n'
             'most frequent amino acid of its reduced group, the group itself is on the line below\n'
             f'circle areas are schematic, counts are exact  ·  {_N:,} sequences in total, '
             f'{n_none:,} in none of the three sets',
             ha='left', va='top', color=INK_2, fontsize=9.5, linespacing=1.55)

    R = 0.5
    CENTRES = [(-0.35, 0.2), (0.35, 0.2), (0.0, -0.4)]
    for (cx, cy), col in zip(CENTRES, SERIES):
        ax.add_patch(Circle((cx, cy), R, facecolor=col, alpha=0.16, lw=0, zorder=1))
        ax.add_patch(Circle((cx, cy), R, facecolor='none', edgecolor=col, lw=2, zorder=3))

    REGION_XY = {'A': (-0.58, 0.44), 'B': (0.58, 0.44), 'AB': (0.0, 0.42), 'C': (0.0, -0.72),
                 'AC': (-0.37, -0.22), 'BC': (0.37, -0.22), 'ABC': (0.0, -0.03)}
    for key, (x, y) in REGION_XY.items():
        ax.text(x, y, f'{region_n[key]:,}', ha='center', va='center', color=INK,
                fontsize=12, fontweight='bold' if key == 'ABC' else 'normal', zorder=4)

    # each label sits outside its own circle, under a swatch in that circle's colour
    LABEL_XY = [(-0.95, 1.04), (0.95, 1.04), (0.0, -1.06)]
    for (x, y), col, name, grp, gof_m, carry_m in zip(
            LABEL_XY, SERIES, view['labels_aa'], view['groups'],
            view['gof_masks'], view['carrier_masks']):
        n_carry, n_gof = int(carry_m.sum()), int(gof_m.sum())
        ax.plot([x - 0.15, x + 0.15], [y + 0.075, y + 0.075], color=col, lw=3, solid_capstyle='round')
        ax.text(x, y, f'{name}\n{grp}', ha='center', va='top', color=INK,
                fontsize=10.5, linespacing=1.45)
        ax.text(x, y - 0.21, f'carriers {n_carry:,} ({100 * n_carry / _N:.1f}%)  ·  gof {n_gof:,}',
                ha='center', va='top', color=INK_2, fontsize=9.5)

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.45, 1.22)
    ax.set_aspect('equal')
    ax.axis('off')
    if save:
        fig.savefig(f"venn_gof_{view['target']}.png", dpi=200, facecolor=SURFACE)
    plt.show()


def plot_sharing(view, save=True, top_n=12):
    """How many contenders are gain of fitness in the same sequence, and which ones."""
    n_shared, frame = view['n_shared'], view['contenders']
    total_gof = int(view['gof_masks'][0].sum())
    n_rows = max(1, min(top_n, int((frame['gof_overlap_with_target'] > 0).sum())))
    # height follows the number of contender rows, so the bars keep the same thickness
    fig, axes = plt.subplots(1, 2, figsize=(12.5, max(3.4, 1.5 + 0.33 * n_rows)), facecolor=SURFACE,
                             gridspec_kw={'width_ratios': [1, 1.35]})

    ax = axes[0]
    ax.set_facecolor(SURFACE)
    ax.set_axisbelow(True)
    ax.grid(axis='y', color='#ececea', lw=0.8)
    if total_gof:
        vals = np.arange(int(n_shared.min()), int(n_shared.max()) + 1)
        counts = np.bincount(n_shared - int(n_shared.min()), minlength=len(vals))
        # one bar per integer, gaps left where a count is genuinely zero
        ax.bar(vals, counts, width=0.85 if len(vals) > 12 else 0.62, color=SERIES[0])
        if len(vals) <= 12:
            ax.set_xticks(vals)
        else:
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        alone = int(counts[0]) if vals[0] == 0 else 0
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.text(0.98, 0.96, f'{alone:,} of {total_gof:,} ({100 * alone / total_gof:.0f}%)\n'
                'have no contender',
                transform=ax.transAxes, ha='right', va='top', fontsize=9.5, color=INK_2,
                linespacing=1.5)
    else:
        ax.text(0.5, 0.5, 'no gain-of-fitness sequences', transform=ax.transAxes,
                ha='center', color=INK_2, fontsize=10)
    ax.set_title(f"{view['target']}: contenders that are gain of fitness in the same sequence",
                 fontsize=11, color=INK, loc='left', pad=12)
    ax.set_xlabel('contending double mutations that are also gain of fitness', fontsize=9.5, color=INK_2)
    ax.set_ylabel(f"gain-of-fitness sequences of {view['target']}", fontsize=9.5, color=INK_2)
    ax.tick_params(colors=INK_2, labelsize=9.5, length=0)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color('#dcdbd6')

    ax = axes[1]
    ax.set_facecolor(SURFACE)
    n_share_any = int((frame['gof_overlap_with_target'] > 0).sum())
    top = frame[frame['gof_overlap_with_target'] > 0].head(top_n).iloc[::-1]
    if len(top):
        y = np.arange(len(top))
        ax.barh(y, top['gof_overlap_with_target'], height=0.68, color=SERIES[0])
        ax.set_yticks(y, [f"{r['pair_aa']}   carriers {r['n_obs']:,}" for _, r in top.iterrows()],
                      fontsize=9.5, color=INK_2)
        span = int(top['gof_overlap_with_target'].max())
        for yi, v in zip(y, top['gof_overlap_with_target']):
            ax.text(v + span * 0.02, yi, f'{v:,}', va='center', fontsize=9.5, color=INK)
        ax.set_xlim(0, span * 1.16)
        ax.set_ylim(-0.7, len(top) - 0.3)
    else:
        ax.text(0.5, 0.5, 'no contender is gain of fitness in any of its sequences',
                transform=ax.transAxes, ha='center', color=INK_2, fontsize=10)
        ax.set_yticks([])
    ax.set_title('gain-of-fitness sequences shared with each contender', fontsize=11,
                 color=INK, loc='left', pad=26)
    ax.text(0, 1.015, f'out of its {total_gof:,} gof sequences  ·  {len(frame):,} contenders, '
            f'{n_share_any} share any', transform=ax.transAxes, fontsize=9.5, color=INK_2, va='bottom')
    ax.tick_params(colors=INK_2, labelsize=9.5, length=0)
    ax.xaxis.set_visible(False)
    for side in ('top', 'right', 'left', 'bottom'):
        ax.spines[side].set_visible(False)

    fig.tight_layout()
    if save:
        fig.savefig(f"gof_sharing_{view['target']}.png", dpi=200, facecolor=SURFACE)
    plt.show()

In [ ]:
view = contender_view('M41L-T215F')
print(f"{len(view['contenders']):,} contenders  ·  regions {view['region_n']}")
print(view['contenders'].head(6).to_string(index=False))
plot_venn(view)
plot_sharing(view)

In [ ]:
# same two figures for another pair -- only the label changes
view_v75 = contender_view('V75M-F77L')
print(f"{len(view_v75['contenders']):,} contenders  ·  regions {view_v75['region_n']}")
print(view_v75['contenders'].head(6).to_string(index=False))
plot_venn(view_v75)
plot_sharing(view_v75)

In [ ]:
# the whole contender list of V75M-F77L, not just the ones drawn: marginal, gain-of-fitness
# count without the contender rule, and how much of its gain-of-fitness set each one takes
v75_contenders = export_contenders(view_v75)
v75_contenders.head(12)

In [ ]:
# to draw named contenders instead of the two that share the most, pass them in -- as amino
# acids or as reduced letters, both are looked up in the contender table
plot_venn(contender_view('M41L-T215F', contenders=['M41L-T215Y', 'D67N-T215Y']), save=False)

## Reading the tables

Per-pair summary (`summary_df`):

| column | meaning |
| --- | --- |
| `n_obs`, `obs_freq`, `w_obs_freq` | bivariate marginal: sequences carrying both mutations, as a count, a fraction, and a weight fraction |
| `n_gof` | gain-of-fitness sequences under the position-local rule, i.e. *without* requiring dE12 to beat the contenders |
| `pct_dmc_gof` | of the sequences that carry the double mutation, the share that are gain of fitness |
| `pct_gof_dmc` | of the gain-of-fitness sequences, the share that actually carry the double mutation |
| `n_gof_shared_with_contender` / `pct_gof_shared` | of those gain-of-fitness sequences, how many are simultaneously gain of fitness for at least one contender -- the double counting the contender constraint removes |
| `mean/max_contender_gof_per_seq` | how many contenders share each gain-of-fitness sequence |
| `n_gof_strict`, `n_gof_lost_to_contenders` | what is left once dE12 must beat every contender, and what that costs |

Per-contender table:

| column | meaning |
| --- | --- |
| `pair_aa`, `pair`, `mutant_group` | the contender as amino acids (most frequent of each reduced group), as reduced letters, and the full group behind each mutant letter |
| `shares_position` | which position of the target it takes with a different residue |
| `n_obs`, `obs_freq`, `w_obs_freq` | its own bivariate marginal |
| `n_gof`, `n_gof_dmc`, `pct_dmc_gof` | its gain-of-fitness sequences (local rule), how many of them carry it, and that as a share of the sequences that carry it |
| `gof_overlap_with_target` | sequences that are gain of fitness for both this contender and the target (on the target's own row this is just its `n_gof`) |
| `n_beats_target` | sequences where the target would be gain of fitness but this contender is the strongest rival and beats it |

`export_contenders(view)` writes the **whole** contender list of one pair to
`contenders_<pair>.csv` -- every double mutation that wants another residue at one of its two
positions, with `n_obs` / `obs_freq` / `w_obs_freq` (the bivariate marginal over the alignment),
`n_gof` (gain of fitness under the plain rule, no contender constraint),
`gof_overlap_with_target`, and that overlap as `pct_of_target_gof` (share of the target's
gain-of-fitness sequences) and `pct_of_own_gof` (share of the contender's own).

Contenders are listed when they beat the target somewhere, or when they are among the most
observed or most frequent gain-of-fitness double mutations at either position. Only conflicting
double mutations count as contenders: one that shares a position with the *same* residue
(M41L-T215F vs M41L-D67N) can hold in the same sequence and is not competing with it.

`n_beats_target` is counted per position, so a sequence whose target pair is beaten at both of
its positions is counted once for each winner; `n_gof_lost_to_contenders` in the summary counts
such a sequence once, which is why the column sums to more than it.